Test Wallahi

In [1]:
import pandas as pd
from preprocessing_Anu import preprocess_solar_data

In [2]:
template_df_solar = preprocess_solar_data("../notebooks/Jacques/Data/Solar_data.csv")

template_df_solar.head()

,datetime,auckland_shortwave_wm2,auckland_sunshine_s,christchurch_shortwave_wm2,christchurch_sunshine_s,wellington_shortwave_wm2,wellington_sunshine_s,hamilton_shortwave_wm2,hamilton_sunshine_s,tauranga_shortwave_wm2,tauranga_sunshine_s,dunedin_shortwave_wm2,dunedin_sunshine_s
0,2014-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2014-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2014-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2014-01-01 03:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2014-01-01 04:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:

# ==========================================
# FINAL CLEANING CODE
# GENERATION DATA PREPROCESSING
# ==========================================

# Loading the Data
df = pd.read_csv("../notebooks/Jacques/Data/generation_output_merged.csv", low_memory = False)

# Cleaning schema renaming in Jan 1st 2020. COnsolidating into Trading Date
df['Trading_date'] = df['Trading_date'].fillna(df['Trading_Date'])
df = df.drop(columns=['Trading_Date'])
df = df.rename(columns={'Trading_date': 'Trading_Date'})

# Unpivot TP Data into multiple rows, each row now corresponds to one date and hour
# Melt from wide to long
# id_vars: columns that stay fixed (will be repeated for each TP)
# value_vars: columns to unpivot (TP1 through TP50)

metadata_cols = ['Site_Code', 'POC_Code', 'Nwk_Code', 'Gen_Code',
                 'Fuel_Code', 'Tech_Code', 'Trading_Date']
tp_cols = [f'TP{i}' for i in range(1, 51)]

df_long = df.melt(
    id_vars=metadata_cols,
    value_vars=tp_cols,
    var_name='TP',
    value_name='generation_kwh'
)

# Drop NaN rows (non-existent trading periods)
df_long = df_long.dropna(subset=['generation_kwh'])

# Extract the TP number as an integer in new column, drop old TP column
df_long['tp_num'] = df_long['TP'].str.replace('TP', '').astype(int)
df_long = df_long.drop(columns=['TP'])

# Convert times to local NZ time, UTC + 12, and true UTC
    # 1. NZ local wall clock = Trading_Date + (tp_num - 1) * 30 minutes
df_long['datetime_nz'] = (
    pd.to_datetime(df_long['Trading_Date'])
    + pd.to_timedelta((df_long['tp_num'] - 1) * 30, unit='m')
)

    # 2. Use pandas timezone tools to make Pandas aware of timezone and
    # derive true UTC and true UTC+12
        # Step A: localize the naive datetime as Pacific/Auckland (handles DST)
nz_tz_aware = df_long['datetime_nz'].dt.tz_localize(
    'Pacific/Auckland',
    nonexistent=pd.Timedelta(hours=1),  # shift DST-gap times forward 1 hour
    ambiguous=True                       # treat DST-end duplicates as first occurrence
)

        # Step B: convert to UTC and UTC+12, then drop the timezone label for clean naive datetimes
df_long['datetime_utc']   = nz_tz_aware.dt.tz_convert('UTC').dt.tz_localize(None)
df_long['datetime_utc12'] = nz_tz_aware.dt.tz_convert('Etc/GMT-12').dt.tz_localize(None)

        # Step C: drop the Trading_Date and tp_num Columns
df_long = df_long.drop(columns=['Trading_Date'])
df_long = df_long.drop(columns=['tp_num'])

# Edit new order
final_order = [
    'datetime_nz', 'datetime_utc12', 'datetime_utc',    # the three datetimes
    'tp_num',                                           # original date + TP for reference
    'Site_Code', 'POC_Code', 'Nwk_Code', 'Gen_Code',
    'Fuel_Code', 'Tech_Code',
    'generation_kwh',
]
df_long = df_long[final_order].sort_values(['datetime_utc12', 'POC_Code']).reset_index(drop=True)


# Average half hour times into one hour. Ex: 1:30 and 2:00 will be averaged into 2:00.
# Centered hourly aggregation
# Each "bucket H" contains TPs at (H-0:30) and H, representing the hour centered on H

df_long['hour_bucket'] = (
    df_long['datetime_utc12'] + pd.Timedelta(minutes=30)
).dt.floor('h')

df_hourly = (
    df_long
    .groupby(
        ['hour_bucket', 'Site_Code', 'POC_Code', 'Nwk_Code',
         'Gen_Code', 'Fuel_Code', 'Tech_Code'],
        observed=True
    )
    .agg(
        datetime_nz    = ('datetime_nz',    'max'),   # 👈 max instead of mean
        datetime_utc12 = ('datetime_utc12', 'max'),   # 👈
        datetime_utc   = ('datetime_utc',   'max'),   # 👈
        generation_kwh = ('generation_kwh', 'mean'),  # still mean for the value
    )
    .reset_index()
    .drop(columns=['hour_bucket'])   # duplicate of datetime_utc12 after the groupby
)

# Reorder columns
final_cols = [
    'datetime_nz', 'datetime_utc12', 'datetime_utc',
    'Site_Code', 'POC_Code', 'Nwk_Code', 'Gen_Code',
    'Fuel_Code', 'Tech_Code', 'generation_kwh',
]
df_hourly = (
    df_hourly[final_cols]                           # 1. column order
    .sort_values(['datetime_utc12', 'POC_Code'])    # 2. row order
    .reset_index(drop=True)                         # 3. clean index
)


# Drop the local NZ tiem and true UTC as it is not needed. DELETE THIS LINE IF WANT TO KEEP
df_hourly = df_hourly.drop(columns=['datetime_nz'])
df_hourly = df_hourly.drop(columns=['datetime_utc'])


# Make UTC + 12 Primary Key


In [32]:
df_hourly.head()

,datetime_utc12,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,generation_kwh
0,2013-12-31 23:00:00,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11360.000
1,2013-12-31 23:00:00,ARG,ARG1101,TRUS,argyle_wairau,Hydro,Hydro,3448.000
2,2013-12-31 23:00:00,ARI,ARI1101,MRPL,arapuni,Hydro,Hydro,19690.000
3,2013-12-31 23:00:00,ARI,ARI1102,MRPL,arapuni,Hydro,Hydro,11233.000
4,2013-12-31 23:00:00,HBK,ASB0661,EASH,highbank,Hydro,Hydro,10985.961


In [27]:
df_long.head()

,datetime_nz,datetime_utc12,datetime_utc,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,generation_kwh
0,2014-01-01,2013-12-31 23:00:00,2013-12-31 11:00:00,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11360.000
1,2014-01-01,2013-12-31 23:00:00,2013-12-31 11:00:00,ARG,ARG1101,TRUS,argyle_wairau,Hydro,Hydro,3448.000
2,2014-01-01,2013-12-31 23:00:00,2013-12-31 11:00:00,ARI,ARI1101,MRPL,arapuni,Hydro,Hydro,19690.000
3,2014-01-01,2013-12-31 23:00:00,2013-12-31 11:00:00,ARI,ARI1102,MRPL,arapuni,Hydro,Hydro,11233.000
4,2014-01-01,2013-12-31 23:00:00,2013-12-31 11:00:00,HBK,ASB0661,EASH,highbank,Hydro,Hydro,10985.961


## Testing Things

In [ ]:
df_long[(df_long['generation_kwh'].isna()) &
        (df_long['TP'] != "TP50") &
        (df_long['TP'] != "TP49") &
        (df_long['Trading_Date'].str[5:7] != '09')]

,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,Trading_Date,TP,generation_kwh


In [ ]:
df_long[(df_long['generation_kwh'].isna()) & (df_long['TP'] != "TP50") & (df_long['TP'] != "TP49")
         & (df_long['TP'] != "TP47") & (df_long['TP'] != "TP48")]

,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,Trading_Date,TP,generation_kwh


In [19]:
df_long[(df_long['Site_Code'] == "ARA") & (df_long['Trading_Date'].str.startswith('2014-06-01'))]

,datetime_nz,datetime_utc12,datetime_utc,Trading_Date,tp_num,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,generation_kwh
515998,2014-06-01 00:00:00,2014-06-01 00:00:00,2014-05-31 12:00:00,2014-06-01,1,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11590.0
516068,2014-06-01 00:30:00,2014-06-01 00:30:00,2014-05-31 12:30:00,2014-06-01,2,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11560.0
516138,2014-06-01 01:00:00,2014-06-01 01:00:00,2014-05-31 13:00:00,2014-06-01,3,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11510.0
516208,2014-06-01 01:30:00,2014-06-01 01:30:00,2014-05-31 13:30:00,2014-06-01,4,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11520.0
516278,2014-06-01 02:00:00,2014-06-01 02:00:00,2014-05-31 14:00:00,2014-06-01,5,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11510.0
516348,2014-06-01 02:30:00,2014-06-01 02:30:00,2014-05-31 14:30:00,2014-06-01,6,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,10970.0
516418,2014-06-01 03:00:00,2014-06-01 03:00:00,2014-05-31 15:00:00,2014-06-01,7,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11040.0
516488,2014-06-01 03:30:00,2014-06-01 03:30:00,2014-05-31 15:30:00,2014-06-01,8,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11070.0
516558,2014-06-01 04:00:00,2014-06-01 04:00:00,2014-05-31 16:00:00,2014-06-01,9,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,10960.0
516628,2014-06-01 04:30:00,2014-06-01 04:30:00,2014-05-31 16:30:00,2014-06-01,10,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,11070.0


In [27]:
df[pd.to_datetime(df['Trading_Date']).dt.year == 2024]

,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,Trading_Date,TP1,TP2,TP3,TP4,TP5,TP6,TP7,TP8,TP9,TP10,TP11,TP12,TP13,TP14,TP15,TP16,TP17,TP18,TP19,TP20,TP21,TP22,TP23,TP24,TP25,TP26,TP27,TP28,TP29,TP30,TP31,TP32,TP33,TP34,TP35,TP36,TP37,TP38,TP39,TP40,TP41,TP42,TP43,TP44,TP45,TP46,TP47,TP48,TP49,TP50
268244,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2024-01-01,10840.0,10840.0,10780.0,10810.0,10830.0,10830.0,10850.0,10840.0,10840.0,10830.0,10750.0,10710.0,10770.0,10870.0,10970.0,10840.0,6650.0,4920.0,4250.0,4220.0,4290.0,4390.0,4310.0,4340.0,4290.0,4280.0,4340.0,4310.0,4310.0,4290.0,4320.0,4290.0,4350.0,4270.0,4350.0,4480.0,4820.0,4810.0,4670.0,4760.0,4870.0,4830.0,4830.0,4780.0,4840.0,4760.0,4870.0,4870.0,NaN,NaN
268245,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2024-01-02,4790.0,4660.0,4870.0,4870.0,4870.0,4880.0,4840.0,4740.0,4870.0,4770.0,4790.0,4760.0,6090.0,6700.0,5810.0,5810.0,5790.0,5820.0,5770.0,5830.0,5820.0,5810.0,5850.0,5830.0,5790.0,5830.0,5790.0,5840.0,5730.0,5780.0,5790.0,5890.0,5850.0,5850.0,5800.0,5790.0,5780.0,5810.0,5800.0,5780.0,6670.0,10310.0,10770.0,10820.0,10820.0,10860.0,10870.0,10800.0,NaN,NaN
268246,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2024-01-03,10800.0,10870.0,10810.0,10840.0,10850.0,10810.0,10800.0,10840.0,10840.0,10790.0,10830.0,10810.0,10840.0,10840.0,10820.0,10820.0,10840.0,10810.0,10850.0,10830.0,10810.0,10830.0,10790.0,10830.0,10780.0,10420.0,10330.0,10340.0,10330.0,10330.0,10340.0,10330.0,10320.0,10330.0,10300.0,10340.0,10260.0,10340.0,10300.0,10350.0,10330.0,10360.0,10660.0,10770.0,10780.0,10810.0,10920.0,10840.0,NaN,NaN
268247,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2024-01-04,10850.0,10750.0,10840.0,10840.0,10790.0,10850.0,10830.0,10750.0,10830.0,10880.0,10890.0,10830.0,10830.0,10780.0,10880.0,10830.0,10800.0,10790.0,10320.0,10320.0,10310.0,10340.0,10340.0,10290.0,10290.0,10340.0,10330.0,10310.0,10360.0,10300.0,10330.0,10310.0,10320.0,10330.0,10320.0,10320.0,10330.0,10310.0,10310.0,10340.0,10340.0,10330.0,10350.0,10290.0,10250.0,10460.0,10460.0,10180.0,NaN,NaN
268248,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2024-01-05,10280.0,10340.0,10380.0,10320.0,10300.0,10380.0,10330.0,10270.0,10350.0,10470.0,10320.0,10260.0,10260.0,10370.0,10350.0,10330.0,10280.0,10330.0,10320.0,10320.0,10290.0,10320.0,10270.0,10330.0,10340.0,10270.0,10250.0,10320.0,10340.0,10330.0,10250.0,10340.0,10330.0,10340.0,10260.0,10330.0,10350.0,12670.0,14290.0,14310.0,14270.0,14310.0,14310.0,14310.0,14300.0,14250.0,14300.0,14290.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298889,WWD,WWD1103,MERI,west_wind,Wind,Wind,2024-12-27,16134.0,17647.0,17850.0,18042.0,17548.0,16725.0,16565.0,15652.0,15489.0,15918.0,14911.0,15434.0,15121.0,14939.0,14840.0,14058.0,15866.0,16030.0,16128.0,16563.0,16416.0,17212.0,17178.0,16728.0,16339.0,17071.0,16606.0,16820.0,16578.0,16639.0,16897.0,16591.0,16981.0,16869.0,16443.0,16850.0,17294.0,17414.0,16942.0,17116.0,17090.0,16949.0,17081.0,16943.0,16833.0,16152.0,15710.0,15378.0,NaN,NaN
298890,WWD,WWD1103,MERI,west_wind,Wind,Wind,2024-12-28,14334.0,13912.0,13403.0,13323.0,12530.0,11857.0,11178.0,11922.0,11259.0,10105.0,9972.0,10023.0,9504.0,8884.0,7773.0,5806.0,5491.0,3950.0,3360.0,2902.0,2651.0,2568.0,1916.0,704.0,687.0,430.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,63.0,597.0,747.0,1457.0,2107.0,2427.0,1816.0,NaN,NaN
298891,WWD,WWD1103,MERI,west_wind,Wind,Wind,2024-12-29,1626.0,1386.0,1683.0,2118.0,2663.0,3346.0,4772.0,6053.0,7219.0,6844.0,9502.0,8007.0,7832.0,8036.0,9308.0,9644.0,9802.0,10019.0,10588.0,10493.0,10892.0,13491.0,15123.0,14540.0,14249.0,13771.0,14444.0,14350.0,15970.0,14722.0,13671.0,16341.0,18029.0,19082.0,19610.0,19338.0,19775.0,20127.0,19224.0,17003.0,14382.0,14265.0,13474.0,13344.0,14384.0,15474.0,15103.0,14644.0,NaN,NaN
298892,WWD,WWD1103,MERI,west_wind,Wind,Wind,2024

In [17]:
pd.set_option('display.max_columns', None)

### Change in April to Summer Hours

In [18]:
df[(df['Site_Code'] == "ARA") & (df['Trading_date'].str.startswith('2014-04'))]

,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,Trading_date,TP1,TP2,TP3,TP4,TP5,TP6,TP7,TP8,TP9,TP10,TP11,TP12,TP13,TP14,TP15,TP16,TP17,TP18,TP19,TP20,TP21,TP22,TP23,TP24,TP25,TP26,TP27,TP28,TP29,TP30,TP31,TP32,TP33,TP34,TP35,TP36,TP37,TP38,TP39,TP40,TP41,TP42,TP43,TP44,TP45,TP46,TP47,TP48,TP49,TP50,Trading_Date
6447,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-01,13120.0,13160.0,13120.0,13260.0,13160.0,13210.0,13260.0,13160.0,13890.0,14160.0,14050.0,14150.0,14020.0,13880.0,13970.0,13480.0,13590.0,13590.0,13510.0,13400.0,13470.0,13570.0,13370.0,13600.0,13400.0,13400.0,13540.0,13390.0,13370.0,13440.0,13440.0,13400.0,13480.0,13530.0,13440.0,13410.0,13560.0,14230.0,14730.0,14710.0,14050.0,13640.0,13700.0,13710.0,13610.0,13630.0,13700.0,13410.0,NaN,NaN,NaN
6518,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-02,13680.0,13540.0,12590.0,12590.0,12630.0,12610.0,12620.0,12760.0,11790.0,10980.0,11210.0,10820.0,12740.0,12810.0,13130.0,12960.0,13130.0,13110.0,13080.0,12950.0,13120.0,12970.0,14610.0,14610.0,14480.0,14490.0,14540.0,14530.0,14400.0,14420.0,14450.0,14460.0,14450.0,14470.0,14110.0,13040.0,13090.0,13080.0,13150.0,13230.0,14030.0,14110.0,13950.0,14090.0,14480.0,14540.0,14530.0,14560.0,NaN,NaN,NaN
6589,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-03,14480.0,14510.0,14480.0,14560.0,14520.0,14480.0,14500.0,14540.0,14540.0,14520.0,14530.0,14530.0,14520.0,14520.0,14540.0,14500.0,14550.0,14530.0,14550.0,14530.0,14410.0,14430.0,14460.0,14460.0,14340.0,14370.0,14390.0,14410.0,14260.0,14300.0,14350.0,14350.0,14360.0,14360.0,13580.0,13230.0,13160.0,13410.0,13350.0,13420.0,13350.0,13520.0,13470.0,13420.0,13450.0,14390.0,14560.0,14560.0,NaN,NaN,NaN
6660,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-04,14550.0,14550.0,14550.0,14540.0,14550.0,14540.0,14530.0,14540.0,14530.0,14530.0,14310.0,14340.0,14310.0,14360.0,14320.0,14360.0,14230.0,14350.0,14350.0,14340.0,14240.0,14140.0,14300.0,14320.0,16830.0,24040.0,23810.0,24160.0,23650.0,23860.0,23670.0,23720.0,18940.0,14350.0,14390.0,14380.0,14390.0,14400.0,14380.0,14550.0,14370.0,14380.0,14380.0,14370.0,14380.0,14370.0,14360.0,14370.0,NaN,NaN,NaN
6731,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-05,12420.0,10700.0,10620.0,12440.0,13290.0,12990.0,10660.0,13690.0,13640.0,13780.0,13660.0,13660.0,11320.0,12250.0,12810.0,12720.0,12720.0,12750.0,12820.0,12710.0,12710.0,12610.0,12770.0,13070.0,14410.0,14440.0,14470.0,14480.0,14350.0,14370.0,14420.0,14420.0,14430.0,14430.0,14450.0,14440.0,14440.0,14460.0,13320.0,11770.0,13660.0,14530.0,14510.0,14520.0,14520.0,14510.0,14520.0,14500.0,NaN,NaN,NaN
6802,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-06,14510.0,14510.0,14500.0,14510.0,14490.0,14500.0,14490.0,14490.0,14480.0,14490.0,14480.0,14500.0,14510.0,14520.0,14510.0,14520.0,14520.0,14510.0,14510.0,14500.0,14510.0,14500.0,14370.0,14390.0,14430.0,14440.0,14300.0,14330.0,14370.0,14380.0,14240.0,14270.0,14320.0,14320.0,14330.0,14330.0,14330.0,14320.0,14320.0,14310.0,13810.0,13820.0,13830.0,13900.0,13770.0,13950.0,13770.0,13880.0,13960.0,13910.0,NaN
6873,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-07,13300.0,12680.0,12500.0,12520.0,11730.0,11640.0,11720.0,11880.0,11830.0,11420.0,11950.0,11530.0,12180.0,13110.0,12940.0,13090.0,13160.0,13270.0,13440.0,13190.0,14450.0,14460.0,14480.0,14500.0,14410.0,14380.0,14440.0,14460.0,14300.0,14370.0,14390.0,14390.0,14360.0,14440.0,14450.0,14450.0,14380.0,14400.0,14430.0,14420.0,14480.0,14440.0,14450.0,14410.0,14430.0,14370.0,14400.0,14410.0,NaN,NaN,NaN
6944,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-08,14390.0,14320.0,14290.0,14400.0,14410.0,14420.0,14430.0,14330.0,14390.0,14380.0,14430.0,14420.0,14440.0,14440.0,14420.0,14460.0,14430.0,14420.0,14480.0,14460.0,14330.0,14360.0,14380.0,13310.0,13120.0,13200.0,13190.0,13070.0,13110.0,13200.0,13240.0,13360.0,13310.0,13470.0,13300.0,13140.0,13270.0,13250.0,13360.0,13270.0,13380.0,13260.0,13300.0,13310.0,13210.0,13110.0,11600.0,11370.0,NaN,NaN,NaN
7015,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-09,1162

In [30]:
df[(df['Site_Code'] == "ARA") & (df['Trading_Date'].str.startswith('2014')) & (df['TP50'].notna())]

,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,Trading_Date,TP1,TP2,TP3,TP4,TP5,TP6,TP7,TP8,TP9,TP10,TP11,TP12,TP13,TP14,TP15,TP16,TP17,TP18,TP19,TP20,TP21,TP22,TP23,TP24,TP25,TP26,TP27,TP28,TP29,TP30,TP31,TP32,TP33,TP34,TP35,TP36,TP37,TP38,TP39,TP40,TP41,TP42,TP43,TP44,TP45,TP46,TP47,TP48,TP49,TP50
6802,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-04-06,14510.0,14510.0,14500.0,14510.0,14490.0,14500.0,14490.0,14490.0,14480.0,14490.0,14480.0,14500.0,14510.0,14520.0,14510.0,14520.0,14520.0,14510.0,14510.0,14500.0,14510.0,14500.0,14370.0,14390.0,14430.0,14440.0,14300.0,14330.0,14370.0,14380.0,14240.0,14270.0,14320.0,14320.0,14330.0,14330.0,14330.0,14320.0,14320.0,14310.0,13810.0,13820.0,13830.0,13900.0,13770.0,13950.0,13770.0,13880.0,13960.0,13910.0


### Change in September to Winter Hours

In [19]:
df[(df['Site_Code'] == "ARA") & (df['Trading_date'].str.startswith('2014-09'))]

,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,Trading_date,TP1,TP2,TP3,TP4,TP5,TP6,TP7,TP8,TP9,TP10,TP11,TP12,TP13,TP14,TP15,TP16,TP17,TP18,TP19,TP20,TP21,TP22,TP23,TP24,TP25,TP26,TP27,TP28,TP29,TP30,TP31,TP32,TP33,TP34,TP35,TP36,TP37,TP38,TP39,TP40,TP41,TP42,TP43,TP44,TP45,TP46,TP47,TP48,TP49,TP50,Trading_Date
17187,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-01,27310.0,27330.0,27290.0,27270.0,27240.0,27260.0,27190.0,27210.0,27180.0,27210.0,27220.0,27220.0,27210.0,27250.0,27270.0,27280.0,27300.0,27350.0,27370.0,27470.0,27330.0,27390.0,27470.0,27490.0,27280.0,27330.0,27410.0,27430.0,27200.0,27260.0,27330.0,27360.0,27380.0,27400.0,27290.0,27370.0,27440.0,27420.0,27420.0,27420.0,22690.0,26380.0,26370.0,27230.0,26850.0,26860.0,27000.0,26980.0,NaN,NaN,NaN
17257,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-02,26940.0,26810.0,26900.0,25940.0,17440.0,17610.0,17540.0,17560.0,17600.0,14330.0,14270.0,14280.0,14280.0,14280.0,14360.0,22280.0,27240.0,27230.0,27100.0,27220.0,26920.0,26990.0,27040.0,27070.0,24390.0,14290.0,14400.0,14450.0,14360.0,14390.0,14440.0,14470.0,14480.0,14500.0,14600.0,16480.0,27050.0,27270.0,27210.0,27220.0,27230.0,27180.0,27210.0,21490.0,14420.0,17010.0,26820.0,26750.0,NaN,NaN,NaN
17327,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-03,26660.0,24790.0,17930.0,14310.0,14350.0,14370.0,14370.0,14390.0,14390.0,14390.0,14400.0,14390.0,14400.0,14400.0,20370.0,26760.0,26570.0,26320.0,26500.0,26440.0,20550.0,14320.0,14430.0,14460.0,14370.0,14390.0,14430.0,14430.0,14320.0,14330.0,14360.0,14360.0,14350.0,14340.0,14330.0,14330.0,14310.0,14310.0,14300.0,14300.0,14310.0,14320.0,14330.0,14340.0,12400.0,11130.0,11050.0,11120.0,NaN,NaN,NaN
17397,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-04,11310.0,10930.0,10940.0,11280.0,11110.0,11280.0,11060.0,11200.0,11110.0,11030.0,11010.0,10970.0,11000.0,11090.0,11220.0,13280.0,14910.0,14940.0,14940.0,14940.0,14820.0,14830.0,14880.0,14900.0,14770.0,14800.0,14840.0,14850.0,14640.0,14700.0,14790.0,14790.0,14810.0,14800.0,14810.0,14840.0,14830.0,14780.0,14880.0,14830.0,14900.0,14840.0,14900.0,14340.0,13430.0,13420.0,13800.0,13550.0,NaN,NaN,NaN
17467,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-05,13760.0,13500.0,13620.0,13670.0,13480.0,13520.0,13660.0,13430.0,13640.0,13530.0,13470.0,14590.0,14800.0,14840.0,14830.0,14870.0,20340.0,26980.0,26960.0,26980.0,26760.0,26680.0,26820.0,24970.0,15820.0,14320.0,14380.0,14380.0,14280.0,14300.0,14340.0,14330.0,14340.0,14320.0,14320.0,14310.0,14310.0,14300.0,14300.0,14300.0,14310.0,14320.0,14320.0,14130.0,14670.0,14690.0,14740.0,14710.0,NaN,NaN,NaN
17537,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-06,14750.0,14640.0,14710.0,14730.0,14730.0,14700.0,14710.0,14660.0,14710.0,14700.0,14650.0,14650.0,14600.0,14670.0,14630.0,14650.0,14630.0,14660.0,14650.0,14690.0,14570.0,14630.0,14690.0,14710.0,14570.0,14650.0,14690.0,14760.0,14620.0,14690.0,14780.0,14780.0,14820.0,14800.0,14830.0,14860.0,14850.0,14770.0,14870.0,14850.0,14760.0,14840.0,14850.0,14770.0,14760.0,14790.0,14740.0,14750.0,NaN,NaN,NaN
17607,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-07,14760.0,14810.0,14780.0,14760.0,14710.0,14780.0,14770.0,14750.0,14780.0,14770.0,14820.0,14730.0,14770.0,14780.0,14740.0,14750.0,14700.0,14690.0,14730.0,14700.0,14560.0,14600.0,14620.0,14610.0,14420.0,14520.0,11770.0,11270.0,11250.0,11220.0,11340.0,11190.0,11300.0,11420.0,11570.0,12070.0,13580.0,14540.0,13020.0,12970.0,13060.0,12320.0,11140.0,11120.0,11190.0,11150.0,11290.0,11180.0,NaN,NaN,NaN
17677,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-08,11320.0,11160.0,11460.0,11250.0,11400.0,11330.0,11390.0,11450.0,11410.0,11380.0,11310.0,11390.0,11510.0,11440.0,13230.0,14410.0,23960.0,27210.0,27150.0,25610.0,15400.0,14370.0,14450.0,14470.0,14390.0,14410.0,14460.0,14480.0,14360.0,14390.0,14430.0,14440.0,16260.0,27100.0,19870.0,26460.0,26540.0,26310.0,26440.0,26430.0,26490.0,26290.0,21900.0,14190.0,14230.0,14250.0,14270.0,14280.0,NaN,NaN,NaN
17747,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-09,143

In [31]:
df[(df['Site_Code'] == "ARA") & (df['Trading_Date'].str.startswith('2014')) & (df['TP47'].isna())]

,Site_Code,POC_Code,Nwk_Code,Gen_Code,Fuel_Code,Tech_Code,Trading_Date,TP1,TP2,TP3,TP4,TP5,TP6,TP7,TP8,TP9,TP10,TP11,TP12,TP13,TP14,TP15,TP16,TP17,TP18,TP19,TP20,TP21,TP22,TP23,TP24,TP25,TP26,TP27,TP28,TP29,TP30,TP31,TP32,TP33,TP34,TP35,TP36,TP37,TP38,TP39,TP40,TP41,TP42,TP43,TP44,TP45,TP46,TP47,TP48,TP49,TP50
19077,ARA,ARA2201,MRPL,aratiatia,Hydro,Hydro,2014-09-28,10950.0,11050.0,10900.0,10870.0,11030.0,10830.0,10880.0,11060.0,10780.0,10970.0,10740.0,11220.0,10820.0,11030.0,10840.0,10960.0,11110.0,10900.0,11910.0,13440.0,13490.0,13500.0,13380.0,13460.0,13430.0,13340.0,13500.0,13380.0,13470.0,13540.0,13340.0,13630.0,13620.0,13810.0,24220.0,25350.0,25190.0,25280.0,25440.0,25140.0,25200.0,25520.0,25220.0,17510.0,14700.0,14250.0,NaN,NaN,NaN,NaN


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 342701 entries, 0 to 342700
Data columns (total 57 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Site_Code     342701 non-null  object 
 1   POC_Code      342701 non-null  object 
 2   Nwk_Code      342701 non-null  object 
 3   Gen_Code      342701 non-null  object 
 4   Fuel_Code     342701 non-null  object 
 5   Tech_Code     342701 non-null  object 
 6   Trading_Date  342701 non-null  object 
 7   TP1           342701 non-null  float64
 8   TP2           342701 non-null  float64
 9   TP3           342701 non-null  float64
 10  TP4           342701 non-null  float64
 11  TP5           342701 non-null  float64
 12  TP6           342701 non-null  float64
 13  TP7           342701 non-null  float64
 14  TP8           342701 non-null  float64
 15  TP9           342701 non-null  float64
 16  TP10          342701 non-null  float64
 17  TP11          342701 non-null  float64
 18  TP12

In [14]:
df_long.isnull().sum()

Site_Code              0
POC_Code               0
Nwk_Code               0
Gen_Code               0
Fuel_Code              0
Tech_Code              0
Trading_Date           0
TP                     0
generation_kwh    685226
dtype: int64

In [14]:
df.select_dtypes(include='object').info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 342701 entries, 0 to 342700
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   Site_Code     342701 non-null  object
 1   POC_Code      342701 non-null  object
 2   Nwk_Code      342701 non-null  object
 3   Gen_Code      342701 non-null  object
 4   Fuel_Code     342701 non-null  object
 5   Tech_Code     342701 non-null  object
 6   Trading_date  153691 non-null  object
 7   Trading_Date  189010 non-null  object
dtypes: object(8)
memory usage: 20.9+ MB


In [ ]:
df.duplicated().sum()

0

In [20]:
# Take first 15% of rows
df_sample = df.head(int(len(df) * 0.15))

# Save to file
df_sample.to_csv("../notebooks/Jacques/Data/df_sample.csv", index=False)